# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [3]:
q('''
SELECT name, country, title
FROM artists
JOIN tracks
USING (artist_id)
''')

,name,country,title
0,Nova Waves,US,Skyline
1,Nova Waves,US,Undertow
2,The Blue Ridge,US,Foothills
3,Kestrel,UK,Aurora
4,Kestrel,UK,Nightfall
5,Marisol,ES,Sol
6,The Blue Ridge,US,Coastline
7,The Blue Ridge,US,Ridgeline
8,Kestrel,UK,Untitled Demo


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [5]:
q('''
SELECT genre, AVG(seconds)
FROM tracks
GROUP BY genre
ORDER BY AVG(seconds) DESC
''')

,genre,AVG(seconds)
0,Electronic,287.5
1,Pop,220.5
2,Latin,210.0
3,Folk,203.0
4,None,150.0


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [24]:
a = q('''
SELECT user, COUNT(play_id) AS plays, COUNT(DISTINCT track_id) AS tracks
FROM plays
GROUP BY user
''')
a
# wrote this query as a callable variable to make wrtiting the assertion easier at the end.

,user,plays,tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [7]:
q('''
SELECT track_id, title
FROM tracks
LEFT JOIN plays
USING (track_id)
WHERE play_id IS NULL
''')

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [8]:
q('''
SELECT name, ROUND(SUM(seconds) / 60, 1) AS minutes
FROM artists
JOIN tracks
USING (artist_id)
JOIN plays
USING (track_id)
GROUP BY name
ORDER BY SUM(seconds) DESC
''')

,name,minutes
0,Kestrel,19.0
1,Nova Waves,14.0
2,The Blue Ridge,6.0
3,Marisol,3.0


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [16]:
q('''
SELECT track_id, title
FROM tracks
''')
# WHERE genre != 'Pop' would have only returned rows where the genre is NOT pop


,track_id,title
0,10,Skyline
1,11,Undertow
2,12,Foothills
3,13,Aurora
4,14,Nightfall
5,15,Sol
6,16,Coastline
7,17,Ridgeline
8,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [17]:
q('''
SELECT played_on, COUNT(play_id) AS plays, COUNT(DISTINCT user) AS users
FROM plays
GROUP BY played_on
ORDER BY played_on
''')

,played_on,plays,users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [23]:
assert len(q('''
SELECT name, country, title
FROM artists
JOIN tracks
USING (artist_id)
''')) == 9, 'Q1 should return one row per track'

assert len(q('''
SELECT track_id, title
FROM tracks
LEFT JOIN plays
USING (track_id)
WHERE play_id IS NULL
''')) == 2, 'Q4: two tracks have never been played'

assert a['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I has the most trouble with q3 and q5, as using the 'COUNT' and 'DISTINCT' functions took a little bit longer to understand. The same issue on q5 with the 'ROUND(SUM)' function. I had to take a bit of time to stop and pause and think about what I actually wanted to extract from the table. I also had to look back into the lecture notebooks to remember how to print a column that was not in the orginial data using the 'AS' function.